# EDA: NU-Hive / OSBH (Gate & Queen State)

**Core Questions:**
1. **The Gate (Bee vs. NoBee):** Can we differentiate actual hive sound from background noise, wind, or traffic using Mel-spectrograms or MFCCs? This validates the input reliability.
2. **Queen State:** Given a validated 'Bee' segment, what is the acoustic signature difference between a Queenright and Queenless hive?

In [ ]:
import os
import glob
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup plotting
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Visualizing the Gate (Bee vs NoBee)
We use `.lab` files to find the intervals of genuine bee sounds versus ambient noise. Here we can plot a Mel-spectrogram side-by-side.

In [ ]:
def plot_mel_spectrogram(audio_path, title="Mel-Spectrogram"):
    """Helper to plot a mel-spectrogram given an audio path."""
    try:
        y, sr = librosa.load(audio_path, sr=16000)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_dB = librosa.power_to_db(S, ref=np.max)
        
        plt.figure()
        librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
        plt.colorbar(format='%+2.0f dB')
        plt.title(title)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not load {audio_path}: {e}")

# Example Usage:
# plot_mel_spectrogram('dataset/nu_hive/sample_bee.wav', title='Bee Sound (Gate Pass)')
# plot_mel_spectrogram('dataset/nu_hive/sample_nobee.wav', title='NoBee / Noise (Gate Fail)')

## 2. Visualizing Queen State
The NU-Hive dataset stores the Queen presence state directly in the filenames (`QueenBee_H1` vs `NoQueenBee_H1`). Once the Gate passes the audio, we want to see if the feature distributions (like MFCCs or Centroid) shift based on queen presence.

In [ ]:
def compare_queen_state_features(df):
    """
    Plots violin plots to compare extracted acoustic features between Queenright 
    and Queenless segments.
    
    Assumes `df` contains columns like 'mfcc_0', 'centroid', and 'has_queen'.
    """
    features_to_plot = ['mfcc_0', 'mfcc_1', 'centroid', 'spread']
    
    # Only plot columns that actually exist in the dataframe
    valid_cols = [col for col in features_to_plot if col in df.columns]
    
    for feature in valid_cols:
        plt.figure()
        sns.violinplot(data=df, x='has_queen', y=feature, palette='Set2')
        plt.title(f'Distribution of {feature} by Queen State')
        plt.xticks(ticks=[0, 1], labels=['Queenless', 'Queenright'])
        plt.show()

# Example Usage:
# df_features = pd.read_csv('dataset/nu_hive/extracted_features.csv')
# compare_queen_state_features(df_features)